In [1]:
import numpy as np
import dask as ds
import earthaccess as ea
import xarray as xr
import sys
print("success")

c:\Users\edmund.gable\.conda\envs\satprocessing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


success


In [2]:
import sys
sys.path.append(r'C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\RESOURCES\\python\\utilities\\bootstrap\\')
print("success")

success


Install library 'resource'
conda install -c conda-forge resource



In [3]:
import pprint
from environment import bootstrap_environment  
pprint.pprint(bootstrap_environment(preferred="edmund_gable"))



print("success")

{'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'hostname': 'NECL04363598',
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 6, 4, 8, 51, 26, 241323),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python/utilities/src/utilities'),
 'workflow_resources': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources')}
success


In [4]:

from utilities.bootstrap.environment import bootstrap_environment
env = bootstrap_environment(preferred="edmund_gable")
env


{'hostname': 'NECL04363598',
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python/utilities/src/utilities'),
 'workflow_resources': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 6, 4, 8, 51, 44, 218094)}

In [5]:
from utilities import get_dates, get_source_file_dates, find_text
find_text("stats")

regex for string stats is: re.compile('stats', re.IGNORECASE)


[{'module': 'dataset_utilities',
  'function': 'get_dataset_products',
  'line_number': 209,
  'line': "dataset_map (str, optional): Specific map type folder to scan (e.g., 'GLOBAL_4KM_DAILY','NES_2KM_STATS')."},
 {'module': 'dataset_utilities',
  'function': 'get_dataset_products',
  'line_number': 225,
  'line': '"PSC": "/.../NES_4KM_STATS/PSC",'},
 {'module': 'dataset_utilities',
  'function': 'resolve_dataset_map',
  'line_number': 334,
  'line': "Explicit map type to filter by (e.g., 'DAILY', 'STATS', 'ANOMS')."},
 {'module': 'dataset_utilities',
  'function': 'resolve_dataset_map',
  'line_number': 428,
  'line': "stats_maps = [m for m in candidate_maps if m[0].endswith('_STATS')]"},
 {'module': 'dataset_utilities',
  'function': 'resolve_dataset_map',
  'line_number': 437,
  'line': "elif period not in ['D', 'DD'] and stats_maps:"},
 {'module': 'dataset_utilities',
  'function': 'resolve_dataset_map',
  'line_number': 439,
  'line': 'provenance_log[\'resolution_steps\'].append(f

Steps for opening Glider Data or Buoy data
- Download data file into github folder 'data'
- In VScode pull datset from data folder into OA_data notebook
- identify structure of file (printout variable names and dimensions (lat lon depth time))
- identify variable names, units (aragonite, pH, carbon concentration, anything else) with index
- Determine criteria of values and variables
    - Time dimension of dates but want a date range
    - ex: January 24 to February 1st
        - specify dates matching time format in datafile
        - or make a broader description if they just give a month (01-31 of month)
    - Space dimension - nail down where it occured with coordinates (lat, long) (*N *E/W) but identify specific region and if its nearshore, offshore
        - with moving data: take all data within a 'box' of coordinates (minimum and maximum of latitude and longitude is bounding box)
        - With buoy data identify specific larger region around fixed point with a centered bounding box (+- 10km for lat and lon)
    - If data does not have saturated aragonite, create function for sat aragonite from pH and CO2 ppm
        - def sat_arag(ph, CO2)
    - For fixed point: Identify mininum saturated aragonite within that date range and identify as the event value
    - For a moving point: identify average of each day within box, then find minimum of all daily averages and identify as event value
    - Identify threshold for an event and create function 
        - If sat_arag < Longfin_threshold:
            - return sat_arag
    - sat_arag>event_data
    - make replicate file with dictionary that can receive all of the sat_arag data over time and space
        - ([start date],[end date], Box:[min longitude][max longitude][min lattitude][max lattitude], [min sat_arag],[below threshold?],[approaching threshold?])

- by saving this data in a table with a known format and NAMED VARIABLES (not hardcoded), you can read in MOM6 data and compare it with a function that runs repeatedly for all events
- output a text file/dictionary with keys and values
- Fill out sheet with dictionary values


Basically same for DO but switch out sat_aragewith mg/L O2, threshold is 2.31+-0.1



In [6]:
import matplotlib.pyplot as plt
import requests

In [7]:
from pathlib import Path

root = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026")
ds = xr.open_dataset(root / "data" / "Raw_Data" / "Vessels" / "vessel_based_bottom_OA_data_2007_2024.nc")

In [10]:
print(ds)

<xarray.Dataset> Size: 211kB
Dimensions:             (time: 977)
Coordinates:
  * time                (time) datetime64[ns] 8kB 2007-07-23 ... 2024-11-10
Data variables:
    data_source         (time) <U14 55kB ...
    cruise              (time) <U11 43kB ...
    obs_type            (time) <U6 23kB ...
    accession           (time) <U7 27kB ...
    lat                 (time) float64 8kB ...
    lon                 (time) float64 8kB ...
    depth               (time) float64 8kB ...
    bottom_depth        (time) float64 8kB ...
    pH_bottom           (time) float64 8kB ...
    omega_bottom        (time) float64 8kB ...
    temperature_bottom  (time) float64 8kB ...
Attributes:
    comment:              Synthesis of bottom pH and aragonite saturation sta...
    data_sources:         CODAP-NA v2021 dataset (https://essd.copernicus.org...
    date_created:         2025-12-01T20:08:49Z
    date_modified:        2025-12-01T20:08:49Z
    time_coverage_start:  2007-07-23
    time_coverage_

In [8]:
print(ds.omega_bottom)


<xarray.DataArray 'omega_bottom' (time: 977)> Size: 8kB
[977 values with dtype=float64]
Coordinates:
  * time     (time) datetime64[ns] 8kB 2007-07-23 2007-07-23 ... 2024-11-10
Attributes:
    units:        1
    long_name:    Bottom Aragonite Saturation State
    description:  Bottom aragonite saturation state is the deepest measuremen...
    comment:      Bottom aragonite saturation state from the original data so...


In [12]:
ds

<xarray.Dataset> Size: 211kB
Dimensions:             (time: 977)
Coordinates:
  * time                (time) datetime64[ns] 8kB 2007-07-23 ... 2024-11-10
Data variables:
    data_source         (time) <U14 55kB 'CODAP_NA_v2021' ... 'NCEI-OCADS'
    cruise              (time) <U11 43kB 'GOMECC1' 'GOMECC1' ... 'PC2406'
    obs_type            (time) <U6 23kB 'Niskin' 'Niskin' ... 'Niskin' 'Niskin'
    accession           (time) <U7 27kB '83633' '83633' ... '0309698' '0309698'
    lat                 (time) float64 8kB 35.96 35.92 35.9 ... 37.71 39.06
    lon                 (time) float64 8kB -74.95 -74.85 ... -74.26 -72.73
    depth               (time) float64 8kB 58.5 96.2 875.0 ... 49.0 97.0 220.0
    bottom_depth        (time) float64 8kB 71.0 110.0 921.0 ... 54.0 103.0 225.0
    pH_bottom           (time) float64 8kB 8.08 8.014 7.994 ... 7.859 7.817
    omega_bottom        (time) float64 8kB 2.34 2.06 1.34 ... 2.17 1.65 1.42
    temperature_bottom  (time) float64 8kB 13.01 12.27 4.458 ... 14.07 12.79
Attributes:
    comment:              Synthesis of bottom pH and aragonite saturation sta...
    data_sources:         CODAP-NA v2021 dataset (https://essd.copernicus.org...
    date_created:         2025-12-01T20:08:49Z
    date_modified:        2025-12-01T20:08:49Z
    time_coverage_start:  2007-07-23
    time_coverage_end:    2024-11-10
    creator_email:        lgarzio@marine.rutgers.edu
    creator_name:         Lori Garzio
    creator_url:          rucool.marine.rutgers.edu
    institution:          Rutgers University
    contributor_name:     Grace Saba,Lori Garzio
    contributor_role:     Principal Investigator,Data Management

In [15]:
ds.omega_bottom.values

array([2.34 , 2.06 , 1.34 , 1.18 , 0.98 , 0.82 , 0.72 , 1.25 , 1.17 ,
       1.24 , 1.23 , 1.24 , 1.29 , 1.55 , 1.66 , 1.73 , 1.65 , 1.34 ,
       1.16 , 1.06 , 0.98 , 0.85 , 0.74 , 0.7  , 0.64 , 1.44 , 1.03 ,
       1.13 , 1.25 , 1.3  , 1.395, 1.15 , 1.33 , 1.23 , 1.74 , 2.025,
       1.45 , 2.115, 1.87 , 1.17 , 1.285, 2.135, 1.97 , 1.665, 1.92 ,
       1.525, 1.94 , 1.99 , 1.41 , 1.57 , 2.49 , 2.36 , 1.49 , 1.13 ,
       1.12 , 0.99 , 0.89 , 0.81 , 0.82 , 1.6  , 1.46 , 1.34 , 1.27 ,
       1.36 , 1.92 , 1.24 , 1.02 , 1.79 , 1.66 , 2.11 , 2.07 , 1.88 ,
       1.39 , 1.33 ,   nan,   nan,   nan,   nan,   nan, 0.9  ,   nan,
       0.7  ,   nan, 0.64 ,   nan,   nan,   nan,   nan,   nan,   nan,
         nan, 1.42 , 1.14 , 1.12 , 1.4  , 1.49 , 1.62 , 1.34 , 1.62 ,
       1.63 , 1.59 , 1.2  , 1.13 , 1.2  , 1.21 , 0.98 , 1.17 , 1.31 ,
       1.51 , 1.19 , 2.11 , 1.27 , 1.6  , 1.45 , 1.9  , 1.26 , 1.76 ,
       1.7  ,   nan, 1.99 , 2.25 , 1.68 , 1.85 , 2.06 , 1.71 , 1.9  ,
       1.64 , 1.29 ,

In [30]:
table_data=ds.to_dataframe()
table_data

,data_source,cruise,obs_type,accession,lat,lon,depth,bottom_depth,pH_bottom,omega_bottom,temperature_bottom
time,,,,,,,,,,,
2007-07-23,CODAP_NA_v2021,GOMECC1,Niskin,83633,35.9640,-74.9535,58.5,71.0,8.0802,2.34,13.010
2007-07-23,CODAP_NA_v2021,GOMECC1,Niskin,83633,35.9162,-74.8502,96.2,110.0,8.0143,2.06,12.272
2007-07-24,CODAP_NA_v2021,GOMECC1,Niskin,83633,35.8982,-74.7913,875.0,921.0,7.9938,1.34,4.458
2007-07-24,CODAP_NA_v2021,GOMECC1,Niskin,83633,35.8323,-74.6477,1608.1,1665.0,7.9913,1.18,3.658
2007-07-24,CODAP_NA_v2021,GOMECC1,Niskin,83633,35.6043,-74.2607,2605.3,2667.0,7.9762,0.98,2.784
...,...,...,...,...,...,...,...,...,...,...,...
2024-11-07,NCEI-OCADS,PC2406,Niskin,0309698,36.0000,-74.7800,371.0,377.0,7.7301,0.94,7.888
2024-11-08,NCEI-OCADS,PC2406,Niskin,0309698,37.9980,-74.9633,20.0,24.0,7.9933,2.20,17.680
2024-11-08,NCEI-OCADS,PC2406,Niskin,0309698,37.8470,-74.5800,49.0,54.0,7.9483,2.17,17.580


In [ ]:
table_data.to_excel(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data\Vessels\vessel_based_bottom_OA_data_2007_2024.xlsx")

Classify omega_arag values and drop values >= the drop threshold (1.4).
- Returns data with lat, lon, depth, pH at each point.

In [9]:
def classify_omega_arag(omega_arag, var_name="omega_arag", drop_above=1.4,
                        status_name="omega_arag_status"):
    import pandas as pd

    # Store original dataset for coordinate extraction
    original_ds = omega_arag if isinstance(omega_arag, xr.Dataset) else None

    if isinstance(omega_arag, xr.Dataset):
        if var_name in omega_arag:
            omega_arag = omega_arag[var_name]
        elif "omega_bottom" in omega_arag:
            omega_arag = omega_arag["omega_bottom"]
        else:
            raise ValueError(f"Dataset does not contain '{var_name}' or 'omega_bottom'")

    if isinstance(omega_arag, pd.DataFrame):
        if var_name not in omega_arag:
            raise ValueError(f"DataFrame does not contain column '{var_name}'")
        selected = omega_arag.loc[omega_arag[var_name] < drop_above].copy()
        selected[status_name] = pd.cut(
            selected[var_name],
            bins=[-float('inf'), 1.0, 1.1, 1.19, drop_above],
            labels=[
                "critical threshold surpassed",
                "Atlantic Sea Scallop threshold surpassed",
                "Atlantic Cod threshold surpassed",
                "suboptimal",
            ],
            right=False,
        ).astype(str)
        return selected

    # Apply filter
    selected = omega_arag.where(omega_arag < drop_above, drop=True)

    status = xr.where(
        selected < 1.0,
        "critical threshold surpassed",
        xr.where(
            selected < 1.1,
            "Atlantic Sea Scallop threshold surpassed",
            xr.where(
                selected < 1.19,
                "Atlantic Cod threshold surpassed",
                "suboptimal",
            ),
        ),
    )

    # Create output dataset with variables
    output_vars = {var_name: selected, status_name: status}
    
    # Add any available coordinates from the original dataset
    output_coords = {}
    if original_ds is not None:
        for coord_name in original_ds.coords:
            if coord_name not in ['time']:  # Skip time if it's a dimension
                try:
                    output_coords[coord_name] = original_ds.coords[coord_name]
                except:
                    pass
        
        # Also add any other data variables that might be useful (pH, etc.)
        for var in original_ds.data_vars:
            if var not in [var_name, status_name]:
                try:
                    # Filter the variable using the same mask
                    filtered_var = original_ds[var].where(omega_arag < drop_above, drop=True)
                    output_vars[var] = filtered_var
                except:
                    pass

    return xr.Dataset(output_vars, coords=output_coords)

Convert classified xarray Dataset to pandas DataFrame with coordinates as columns.

In [10]:

# Helper function to convert classification results to dataframe with all coordinates as columns
def classification_to_dataframe(classified_ds):
    df = classified_ds.to_dataframe().reset_index()
    return df


In [11]:
OA_events=classify_omega_arag(ds, var_name="omega_bottom")

In [12]:
OA_events.to_dataframe

<bound method Dataset.to_dataframe of <xarray.Dataset> Size: 95kB
Dimensions:             (time: 370)
Coordinates:
  * time                (time) datetime64[ns] 3kB 2007-07-24 ... 2024-11-07
Data variables:
    omega_bottom        (time) float64 3kB 1.34 1.18 0.98 ... 0.93 1.05 0.94
    omega_arag_status   (time) <U40 59kB 'suboptimal' ... 'critical threshold...
    data_source         (time) object 3kB 'CODAP_NA_v2021' ... 'NCEI-OCADS'
    cruise              (time) object 3kB 'GOMECC1' 'GOMECC1' ... 'PC2406'
    obs_type            (time) object 3kB 'Niskin' 'Niskin' ... 'Niskin'
    accession           (time) object 3kB '83633' '83633' ... '0309698'
    lat                 (time) float64 3kB 35.9 35.83 35.6 ... 42.35 42.31 36.0
    lon                 (time) float64 3kB -74.79 -74.65 ... -70.28 -74.78
    depth               (time) float64 3kB 875.0 1.608e+03 ... 27.0 371.0
    bottom_depth        (time) float64 3kB 921.0 1.665e+03 ... 32.0 377.0
    pH_bottom           (time) float

In [34]:
table=classification_to_dataframe(OA_events)
table
table.to_excel(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Processed_Data\2vessel_based_bottom_OA_data_classified.xlsx", index=False)

Average omega_arag by day and classify each day based on the average value.
- Returns a DataFrame with date, average omega_arag, and status.

In [13]:
def average_OA_by_day(omega_arag, var_name="omega_arag", drop_above=1.4):
    import pandas as pd
    import xarray as xr

    if isinstance(omega_arag, xr.Dataset):
        if var_name in omega_arag:
            omega_arag = omega_arag[var_name]
        elif "omega_bottom" in omega_arag:
            omega_arag = omega_arag["omega_bottom"]
        else:
            raise ValueError(f"Dataset does not contain '{var_name}' or 'omega_bottom'")

    if isinstance(omega_arag, xr.DataArray):
        df = omega_arag.to_dataframe(name=var_name).reset_index()
    elif isinstance(omega_arag, pd.DataFrame):
        df = omega_arag.copy()
        if var_name not in df.columns:
            if df.shape[1] == 1:
                df = df.rename(columns={df.columns[0]: var_name})
            else:
                raise ValueError(f"DataFrame does not contain column '{var_name}'")
    else:
        raise ValueError("omega_arag must be an xarray Dataset/DataArray or a pandas DataFrame")

    if 'time' not in df.columns:
        raise ValueError("Input data must contain a 'time' column or coordinate for averaging by day")

    df['time'] = pd.to_datetime(df['time'])

    def find_column(candidates):
        for col in candidates:
            if col in df.columns:
                return col
        return None

    latitude_col = find_column(['latitude', 'lat', 'Latitude', 'LATITUDE'])
    longitude_col = find_column(['longitude','lat', 'lon', 'Longitude', 'LON'])
    ph_col = find_column(['pH', 'ph', 'PH'])
    depth_col = find_column(['depth', 'Depth', 'DEPTH'])
    cruise_col = find_column(['cruise', 'Cruise', 'CRUISE'])
    datasource_col = find_column(['datasource', 'source', 'data_source', 'data source'])
    accession_col = find_column(['accession', 'Accession', 'ACCESSION'])
    obs_type_col = find_column(['obs_type', 'obstype', 'obs type', 'OBS_TYPE'])

    agg_dict = {var_name: 'mean'}
    if latitude_col is not None:
        agg_dict[latitude_col] = ['min', 'max']
    if longitude_col is not None:
        agg_dict[longitude_col] = ['min', 'max']
    if ph_col is not None:
        agg_dict[ph_col] = 'mean'
    if depth_col is not None:
        agg_dict[depth_col] = 'mean'
    if cruise_col is not None:
        agg_dict[cruise_col] = 'first'
    if datasource_col is not None:
        agg_dict[datasource_col] = 'first'
    if accession_col is not None:
        agg_dict[accession_col] = 'first'
    if obs_type_col is not None:
        agg_dict[obs_type_col] = 'first'

    daily_avg = df.groupby(df['time'].dt.date).agg(agg_dict).reset_index()
    daily_avg.rename(columns={'time': 'date'}, inplace=True)

    if isinstance(daily_avg.columns, pd.MultiIndex):
        daily_avg.columns = ['_'.join(col).strip('_') for col in daily_avg.columns.values]

    rename_map = {
        var_name: 'average_omega',
        ph_col: 'average_pH',
        depth_col: 'average_depth',
        f'{latitude_col}_min' if latitude_col else None: 'min_latitude',
        f'{latitude_col}_max' if latitude_col else None: 'max_latitude',
        f'{longitude_col}_min' if longitude_col else None: 'min_longitude',
        f'{longitude_col}_max' if longitude_col else None: 'max_longitude',
        cruise_col: 'cruise',
        datasource_col: 'datasource',
        accession_col: 'accession',
        obs_type_col: 'obs_type',
    }
    rename_map = {k: v for k, v in rename_map.items() if k is not None}
    daily_avg = daily_avg.rename(columns=rename_map)

    daily_avg['status'] = pd.cut(
        daily_avg['average_omega'],
        bins=[-float('inf'), 1.0, 1.1, 1.19, drop_above],
        labels=[
            "critical threshold surpassed",
            "Atlantic Sea Scallop threshold surpassed",
            "Atlantic Cod threshold surpassed",
            "suboptimal",
        ],
        right=False,
    ).astype(str)

    daily_avg = daily_avg.dropna(subset=['status'])
    daily_avg = daily_avg[daily_avg['status'] != 'nan']

    return daily_avg

In [30]:
average_OA_by_day(ds, var_name="omega_bottom")

,date,average_omega,status
1,2007-07-24,1.080000,Atlantic Sea Scallop threshold surpassed
2,2007-07-25,0.720000,critical threshold surpassed
3,2007-07-29,1.226000,suboptimal
6,2007-08-01,1.012500,Atlantic Sea Scallop threshold surpassed
7,2007-08-02,0.693333,critical threshold surpassed
...,...,...,...
375,2024-08-13,1.345000,suboptimal
376,2024-08-14,1.380000,suboptimal
378,2024-08-16,1.355000,suboptimal
381,2024-08-19,1.240000,suboptimal


In [31]:
daily_OA_events = average_OA_by_day(ds, var_name="omega_bottom")
daily_OA_events


,date,average_omega,status
1,2007-07-24,1.080000,Atlantic Sea Scallop threshold surpassed
2,2007-07-25,0.720000,critical threshold surpassed
3,2007-07-29,1.226000,suboptimal
6,2007-08-01,1.012500,Atlantic Sea Scallop threshold surpassed
7,2007-08-02,0.693333,critical threshold surpassed
...,...,...,...
375,2024-08-13,1.345000,suboptimal
376,2024-08-14,1.380000,suboptimal
378,2024-08-16,1.355000,suboptimal
381,2024-08-19,1.240000,suboptimal


In [49]:
daily_OA_events.to_excel(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Processed_Data\vessel_based_bottom_OA_daily_average_classified.xlsx", index=False)

In [50]:
dsg=xr.open_dataset(r"C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\GitHub\\MOM6-COBALT_2026\\data\\Raw_Data\\Gliders\\glider_based_bottom_OA_data_2019_2025 (1).nc")
dsg
dsg.to_dataframe()

,deployment,lat,lon,depth,bottom_depth,pH_bottom,omega_bottom,temperature_bottom
time,,,,,,,,
2019-08-05 00:01:15.711111168,ru30-20190717T1812,39.809955,-73.212830,39.491982,43.0,7.8281,1.08,7.950000
2019-08-05 00:05:55.698529024,ru30-20190717T1812,39.810494,-73.214054,39.546547,43.0,7.8257,1.08,7.950000
2019-08-05 00:10:47.911289856,ru30-20190717T1812,39.811057,-73.215331,41.148884,43.0,7.8294,1.09,7.960000
2019-08-05 00:15:05.200000000,ru30-20190717T1812,39.811553,-73.216456,41.044708,43.0,7.8278,1.08,7.960000
2019-08-05 00:19:32.785124096,ru30-20190717T1812,39.812068,-73.217626,42.493248,43.0,7.8300,1.09,7.970000
...,...,...,...,...,...,...,...,...
2025-08-11 14:30:33.000000000,ru39-20250716T1542,39.305976,-74.335720,12.836317,15.0,8.0142,2.69,23.660000
2025-08-11 14:45:56.144999936,ru39-20250716T1542,39.305388,-74.335010,12.325262,15.0,8.0202,2.72,23.670000
2025-08-11 14:46:08.020999936,ru39-20250716T1542,39.305386,-74.335017,12.196258,15.0,8.0215,2.72,23.670000


In [ ]:
glider_data_2019_2025 = classify_omega_arag(dsg, var_name="omega_bottom")
table_data_glider = glider_data_2019_2025.to_dataframe()

# FIX: Force pandas to modify the existing dataframe
table_data_glider.reset_index(inplace=True)

print(table_data_glider)

table_data_glider.to_csv(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Processed_Data\glider_based_bottom_OA_data_2019_2025_classified.csv", index=False)


                               time  omega_bottom  \
0     2019-08-05 00:01:15.711111168          1.08   
1     2019-08-05 00:05:55.698529024          1.08   
2     2019-08-05 00:10:47.911289856          1.09   
3     2019-08-05 00:15:05.200000000          1.08   
4     2019-08-05 00:19:32.785124096          1.09   
...                             ...           ...   
57571 2025-08-04 19:14:47.456999936          1.34   
57572 2025-08-04 19:45:23.457999872          1.39   
57573 2025-08-04 19:45:30.457999872          1.37   
57574 2025-08-08 15:28:22.362999808          1.40   
57575 2025-08-08 21:10:39.416000000          1.40   

                              omega_arag_status          deployment  \
0      Atlantic Sea Scallop threshold surpassed  ru30-20190717T1812   
1      Atlantic Sea Scallop threshold surpassed  ru30-20190717T1812   
2      Atlantic Sea Scallop threshold surpassed  ru30-20190717T1812   
3      Atlantic Sea Scallop threshold surpassed  ru30-20190717T1812   
4      A

In [ ]:
glider_data_2019_2025=classify_omega_arag(dsg, var_name="omega_bottom")
glider_data_2019_2025
table_data_glider=glider_data_2019_2025.to_dataframe()
table_data_glider.reset_index()
table_data_glider['time'].
print(table_data_glider)
table_data_glider.to_csv(r"C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\GitHub\\MOM6-COBALT_2026\\data\\Processed_Data\\glider_based_bottom_OA_data_2019_2025_classified.csv", index=False)



                               omega_bottom  \
time                                          
2019-08-05 00:01:15.711111168          1.08   
2019-08-05 00:05:55.698529024          1.08   
2019-08-05 00:10:47.911289856          1.09   
2019-08-05 00:15:05.200000000          1.08   
2019-08-05 00:19:32.785124096          1.09   
...                                     ...   
2025-08-04 19:14:47.456999936          1.34   
2025-08-04 19:45:23.457999872          1.39   
2025-08-04 19:45:30.457999872          1.37   
2025-08-08 15:28:22.362999808          1.40   
2025-08-08 21:10:39.416000000          1.40   

                                                      omega_arag_status  \
time                                                                      
2019-08-05 00:01:15.711111168  Atlantic Sea Scallop threshold surpassed   
2019-08-05 00:05:55.698529024  Atlantic Sea Scallop threshold surpassed   
2019-08-05 00:10:47.911289856  Atlantic Sea Scallop threshold surpassed   
2019-08-05 00

In [45]:
print(len(np.unique(table_data_glider["deployment"])))
np.unique(table_data_glider["deployment"])

35


array(['SBU01-20210513T1921', 'SBU01-20210723T1544',
       'SBU01-20220805T1855', 'SBU01-20221011T1924',
       'SBU01-20230706T1707', 'SBU01-20231011T1805',
       'SBU01-20240212T1652', 'SBU02-20230420T1743',
       'SBU02-20230803T1546', 'SBU02-20231018T1829',
       'SBU02-20250513T1539', 'ru30-20190717T1812', 'ru30-20210716T1804',
       'ru30-20220906T1523', 'ru32-20240723T1443', 'ru32-20250716T1541',
       'ru34-20230920T1506', 'ru39-20230420T1636', 'ru39-20230817T1520',
       'ru39-20231103T1413', 'ru39-20240429T1522', 'ru39-20240723T1442',
       'ru39-20241021T1717', 'ru39-20250226T1700', 'ru39-20250423T1535',
       'ru39-20250716T1542', 'ru43-20240612T1658', 'ru43-20240904T1539',
       'um_242-20210630T1904', 'um_242-20211105T1545',
       'um_242-20220513T1556', 'um_242-20220903T1649',
       'um_242-20221205T1648', 'um_242-20230803T1949',
       'um_242-20231109T1526'], dtype=object)

In [ ]:
import PyCO2SYS as pyco2

{'opt_k_bisulfate': 1, 'opt_k_carbonic': 10, 'opt_k_fluoride': 1, 'opt_total_borate': 1, 'opt_gas_constant': 3, 'opt_pH_scale': 1, 'opt_buffers_mode': 1, 'salinity': array(32.), 'temperature': np.float64(25.0), 'pressure': np.float64(0.0), 'total_ammonia': np.float64(0.0), 'total_borate': np.float64(380.0685714285715), 'total_calcium': np.float64(9403.035155062604), 'total_fluoride': np.float64(62.46933914365007), 'total_phosphate': np.float64(0.0), 'total_silicate': np.float64(0.0), 'total_sulfate': np.float64(25815.254064329256), 'total_sulfide': np.float64(0.0), 'peng_correction': np.float64(0.0), 'gas_constant': array(83.14462618), 'total_alpha': 0.0, 'total_beta': 0.0, 'pressure_atmosphere': np.float64(1.0), 'pressure_atmosphere_out': np.float64(1.0), 'opt_pressured_kCO2': 0, 'opt_adjust_temperature': 1, 'opt_which_fCO2_insitu': 1, 'bh_upsilon': np.float64(28995.0), 'k_CO2': array(0.02883839), 'k_carbonic_1': np.float64(1.384641373786189e-06), 'k_carbonic_2': np.float64(1.00732754